# Séance 9 — Prévoir

**Avec les moindres carrés — Analyse des données, L3 Économie**

Depuis huit séances vous cherchez à **expliquer**. Aujourd'hui on change de question, sans changer d'outil.

La semaine prochaine, on ne changera *que* l'estimateur — et le protocole installé aujourd'hui ne bougera pas d'une ligne.

> *Exécution → Tout exécuter* avant de démarrer.


## 0. Les données du TD 9

In [ ]:
!pip install -q yfinance

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import yfinance as yf

TITRE, MARCHE = "RNO.PA", "^FCHI"

brut = yf.download([TITRE, MARCHE], start="2019-01-01", end="2025-12-31",
                   auto_adjust=True, progress=False)
px  = brut["Close"].dropna()
vlm = brut["Volume"].dropna()
px.columns  = ["MARCHE" if c == MARCHE else "TITRE" for c in px.columns]
vlm.columns = ["MARCHE" if c == MARCHE else "TITRE" for c in vlm.columns]

r = np.log(px).diff().dropna()
print(px.shape, "| du", px.index.min().date(), "au", px.index.max().date())

## 1. Le piège : juger un modèle sur son $R^2$

Régressons le prix de demain sur le prix d'aujourd'hui, et comparons à une règle sans aucun paramètre.


In [ ]:
p = pd.DataFrame({"demain": px["TITRE"].shift(-1), "aujourdhui": px["TITRE"]}).dropna()
mod = smf.ols("demain ~ aujourdhui", data=p).fit()

err_modele = np.sqrt(mean_squared_error(p["demain"], mod.fittedvalues))
err_naif   = np.sqrt(mean_squared_error(p["demain"], p["aujourdhui"]))

print("R2 du modele                   :", round(mod.rsquared, 5))
print("erreur du modele estime        :", round(err_modele, 4), "EUR")
print("erreur de 'demain = aujourdhui':", round(err_naif, 4), "EUR")
print("gain du modele                 :", round(100 * (1 - err_modele / err_naif), 2), "%")

Le $R^2$ ne mesurait pas la qualité de la prévision : il mesurait la **persistance de la série**.

> **Un modèle de prévision n'existe pas sans son repère** — et le repère doit être la règle la plus bête qui réponde à la même question.

## 2. Le protocole

Apprentissage sur le passé, test sur l'avenir, évaluation contre un repère.


In [ ]:
X = pd.DataFrame(index=px.index)
for k in range(1, 6):
    X[f"r_lag{k}"] = r["TITRE"].shift(k)
X["ma5"]     = r["TITRE"].shift(1).rolling(5).mean()
X["ma21"]    = r["TITRE"].shift(1).rolling(21).mean()
X["vol5"]    = r["TITRE"].shift(1).rolling(5).std()
X["vol21"]   = r["TITRE"].shift(1).rolling(21).std()
X["marche1"] = r["MARCHE"].shift(1)
X["lvolume"] = np.log(vlm["TITRE"]).shift(1)
VARS = list(X.columns)

def jeu(cible, extra=None):
    """Assemble cible + variables, puis coupe 80 / 20 DANS LE TEMPS."""
    bloc = [cible.rename("y"), X] + ([extra] if extra is not None else [])
    d = pd.concat(bloc, axis=1).dropna()
    c = int(len(d) * 0.8)
    return d.iloc[:c], d.iloc[c:]

def r2_hors(y, prev, repere):
    y = np.asarray(y)
    return 1 - np.sum((y - prev)**2) / np.sum((y - repere)**2)

tr, te = jeu(r["TITRE"])
print(len(VARS), "variables")
print("apprentissage :", tr.index[0].date(), "->", tr.index[-1].date(), f"({len(tr)})")
print("test          :", te.index[0].date(), "->", te.index[-1].date(), f"({len(te)})")

Le découpage est **chronologique**. Découper au hasard entraînerait le modèle sur 2025 pour prédire 2021 — et comme les variables à fenêtres glissantes se chevauchent, le mélange placerait dans l'apprentissage des observations quasi identiques à celles du test.

Chaque variable porte un `.shift(1)` : elle doit être **disponible la veille au soir**.

## 3. Le $R^2$ hors échantillon, et son signe

$$R^2_{\text{hors}} = 1 - \frac{\sum (y - \hat{y})^2}{\sum (y - \text{repère})^2}$$

Le repère n'est pas la moyenne du test — on ne la connaît pas au moment de prévoir.


In [ ]:
m = LinearRegression().fit(tr[VARS], tr["y"])
prev = m.predict(te[VARS])

print("R2 en APPRENTISSAGE :", round(m.score(tr[VARS], tr["y"]), 4))
print()
print("R2 HORS ECHANTILLON (repere = zero)    :",
      round(r2_hors(te["y"], prev, 0), 4))
print("R2 HORS ECHANTILLON (repere = moyenne) :",
      round(r2_hors(te["y"], prev, tr["y"].mean()), 4))

Onze variables sur plus de mille journées donnent **forcément** un $R^2$ d'apprentissage positif : les moindres carrés exploitent le moindre hasard. Hors échantillon, il est indistinguable de zéro.

Un $R^2$ hors échantillon **négatif** n'est pas une erreur de calcul : le modèle fait pire que le repère. L'écart entre les deux chiffres, c'est exactement le **surajustement**.

## 4. Le surajustement n'attend pas les modèles compliqués

Prenons la cible la plus facile — l'amplitude — et ajoutons à la régression des variables de **bruit pur**.


In [ ]:
rng = np.random.default_rng(0)
cible = r["TITRE"].abs()

base = pd.concat([cible.rename("y"), X], axis=1).dropna()
bruit = pd.DataFrame(rng.normal(size=(len(base), 800)),
                     columns=[f"b{i}" for i in range(800)], index=base.index)
d = pd.concat([base, bruit], axis=1)
c = int(len(d) * 0.8)
a, b = d.iloc[:c], d.iloc[c:]

ks = [0, 5, 10, 25, 50, 100, 200, 400, 600, 800]
app, test, nvar = [], [], []
for k in ks:
    cols = VARS + [f"b{i}" for i in range(k)]
    mm = LinearRegression().fit(a[cols], a["y"])
    app.append(mm.score(a[cols], a["y"]))
    test.append(r2_hors(b["y"], mm.predict(b[cols]), a["y"].mean()))
    nvar.append(len(cols))

fig, ax = plt.subplots(figsize=(7.8, 3))
ax.plot(nvar, app,  "o-", color="#3D4543", lw=2, ms=4)
ax.plot(nvar, test, "o-", color="#E2162F", lw=2, ms=4)
ax.text(nvar[-1] + 20, app[-1],  "apprentissage", color="#3D4543", va="center")
ax.text(nvar[-1] + 20, test[-1], "test",          color="#E2162F", va="center")
ax.axhline(0, color="#9AA0A0", ls=":", lw=1)
ax.set_xlabel("Nombre de variables  (les ajoutées sont du bruit pur)")
ax.set_ylabel("$R^2$"); ax.set_xlim(-20, nvar[-1] + 210)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print(f"{nvar[0]:>4} variables : apprentissage {app[0]:.3f}   test {test[0]: .3f}")
print(f"{nvar[-1]:>4} variables : apprentissage {app[-1]:.3f}   test {test[-1]: .3f}")

Aucun arbre, aucun réseau de neurones : des **moindres carrés ordinaires**. Le $R^2$ d'apprentissage monte sans limite, celui de test s'effondre et devient négatif.

Le surajustement n'est pas une maladie des modèles compliqués. C'est la conséquence d'avoir plus de souplesse que de signal.

## 5. Prévoir seulement le sens


In [ ]:
hausse = (te["y"] > 0).mean()
juste  = ((prev > 0) == (te["y"] > 0)).mean()
n      = len(te)
marge  = 1.96 * 0.5 / np.sqrt(n)

print("seances en hausse dans le test :", round(100 * hausse, 1), "%")
print("  -> 'toujours en hausse' a raison", round(100 * hausse, 1), "% du temps")
print("le modele a raison             :", round(100 * juste, 1), "%")
print()
print(f"incertitude sur {n} journees : +/- {100 * marge:.1f} points")

Trois nombres, jamais un seul : le score du modèle, le **taux de base**, et la **marge d'erreur**. Un modèle à 52 % contre un taux de base à 50 % n'a rien démontré.

C'est l'erreur la plus répandue dans les projets étudiants sur données financières.

## 6. Et pourtant, quelque chose est prévisible

Même code, mêmes variables, même découpage. On change **seulement la cible**.


In [ ]:
for cible, nom in [(r["TITRE"], "rendement"), (r["TITRE"].abs(), "|rendement| (amplitude)")]:
    a_, b_ = jeu(cible)
    mm = LinearRegression().fit(a_[VARS], a_["y"])
    print(f"{nom:<24} apprentissage {mm.score(a_[VARS], a_['y']): .3f}"
          f"   hors echantillon {r2_hors(b_['y'], mm.predict(b_[VARS]), a_['y'].mean()): .3f}")

Le **sens** ne peut pas être prévisible : s'il l'était, on achèterait, le prix monterait, l'occasion disparaîtrait. Un $R^2$ nul sur les rendements n'est pas un échec — c'est une **confirmation** de l'efficience.

L'**amplitude** n'offre aucun gain direct : savoir que demain sera agité ne dit pas dans quel sens. Rien ne pousse à l'arbitrer, la régularité subsiste. C'est le regroupement de volatilité du TD.

Toute la gestion du risque vit de cette asymétrie.

## 7. La fuite d'information


In [ ]:
def essai(avec_shift):
    Z = X.copy()
    s = 1 if avec_shift else 0
    Z["vol5"] = r["TITRE"].shift(s).rolling(5).std()      # la seule ligne qui change
    d_ = pd.concat([r["TITRE"].abs().rename("y"), Z], axis=1).dropna()
    c_ = int(len(d_) * .8)
    a_, b_ = d_.iloc[:c_], d_.iloc[c_:]
    mm = LinearRegression().fit(a_[VARS], a_["y"])
    return r2_hors(b_["y"], mm.predict(b_[VARS]), a_["y"].mean())

print("vol5 correcte, avec .shift(1) :", round(essai(True), 3))
print("vol5 SANS le .shift(1)        :", round(essai(False), 3))

La volatilité « des 5 derniers jours » calculée **jour $t$ inclus** contient un cinquième de la cible. Le modèle ne prévoit pas : il **recopie**.

Et surtout : **aucune erreur, aucun avertissement**. Le seul signal est une performance trop belle. Le découpage chronologique ne protège pas de cette fuite-là — les deux précautions sont indépendantes.

## 8. Ce qu'il faut retenir

1. Un $R^2$ calculé sur les données d'apprentissage ne dit rien de la prévision.
2. Un modèle de prévision se compare à un **repère**, toujours.
3. Sur une série temporelle, le découpage est **chronologique**, et chaque variable doit être disponible la veille.
4. Le surajustement est l'affaire de **tous** les modèles, la régression comprise.
5. Un échec de prévision bien mesuré est un **résultat**.

**Séance 10** — arbre de décision et forêt aléatoire, sur les mêmes données, le même découpage et les mêmes repères. On ne changera que l'estimateur.
